# Notebook 02 — Reconstructor (baseline)

Train the **reconstructor (AR)**: given a short English description `z`, predict the activation `h` that produced it.

Architecture (per the paper):

```
z (text)  ──►  [Pythia layers 0..6, FROZEN]  ──►  last-token h_l  ──►  [Linear 768→768]  ──►  ĥ
```

We don't have a verbalizer yet, so for this baseline `z` is a **placeholder**: the first 16 tokens of the snippet that produced `h`. This is a crude hand-rolled warm-start — far worse than what a learned verbalizer should eventually do, but enough to establish:

1. The reconstructor pipeline works end-to-end.
2. The training loop converges.
3. A baseline FVE number on the board, that we'll beat once the real verbalizer is plugged in.

**Requires:** `data/activations/pythia160m_layer6_wikitext2_5000.pt` + `.jsonl` from notebook 01.

## 0. Environment

In [ ]:
# Install deps. Colab's GPU runtime ships a stale sympy that crashes
# when PyTorch loads torch._dynamo — so we upgrade it and, if needed,
# bounce the kernel so torch will see the fresh version.
%pip install -q transformers==4.46.0 accelerate matplotlib "sympy>=1.13.3"

try:
    import sympy.printing.precedence  # the specific submodule torch._dynamo needs
    import sympy
    print(f'sympy {sympy.__version__} OK — no restart needed.')
except Exception:
    print('Restarting runtime to pick up the upgraded sympy...')
    print('After Colab reconnects, click Runtime > Run all again from the top.')
    import os
    os.kill(os.getpid(), 9)

In [ ]:
import json, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Persistent storage: mount Google Drive on Colab so saved artifacts survive
# session resets. On local Python, just use the project root.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/kth-nla-2026')
    print(f'persistent storage: {DATA_ROOT}')
except ImportError:
    DATA_ROOT = Path('.')
    print(f'local storage: {DATA_ROOT.resolve()}')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Config

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-160m'
LAYER = 6
ACT_PT   = DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.pt'
ACT_META = DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.jsonl'

DESC_TOKENS = 16     # placeholder description length
TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3

CKPT_DIR = DATA_ROOT / 'checkpoints'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = DATA_ROOT / 'figures';     FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the activations harvested by notebook 01

In [ ]:
assert ACT_PT.exists(), f'Missing {ACT_PT}. Run notebook 01 first.'

acts = torch.load(ACT_PT, map_location='cpu')
with open(ACT_META, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

assert acts.shape[0] == len(records), 'activation/metadata count mismatch'
print(f'loaded {acts.shape[0]} activations of dim {acts.shape[1]}')
print(f'example record: {records[0]["snippet"][:80]!r}')

## 3. Build placeholder descriptions

For each `(h, snippet)`, the description `z` is the **first `DESC_TOKENS` tokens** of the snippet, re-decoded. This is the crude warm-start. Cheap to compute, deterministic, and matches the spirit of "a short summary of the activation's context."

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token  # Pythia has no pad token by default

def make_description(snippet: str, n_tokens: int = DESC_TOKENS) -> str:
    ids = tok(snippet, return_tensors='pt', truncation=True, max_length=n_tokens).input_ids[0]
    return tok.decode(ids, skip_special_tokens=True)

descriptions = [make_description(r['snippet']) for r in records]
print('example placeholder description:')
print(' ', descriptions[0])

## 4. Train / val / test split

Fixed-seed shuffle, then slice.

In [ ]:
N = acts.shape[0]
assert TRAIN_N + VAL_N + TEST_N <= N

perm = torch.randperm(N, generator=torch.Generator().manual_seed(SEED)).tolist()
split = {
    'train': perm[:TRAIN_N],
    'val':   perm[TRAIN_N:TRAIN_N + VAL_N],
    'test':  perm[TRAIN_N + VAL_N:TRAIN_N + VAL_N + TEST_N],
}
for name, ids in split.items():
    print(f'{name:5}: {len(ids)}')

In [ ]:
class ActDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        return {'z': descriptions[j], 'h': acts[j]}

def collate(batch):
    return {
        'z': [b['z'] for b in batch],
        'h': torch.stack([b['h'] for b in batch]),
    }

loaders = {
    name: DataLoader(ActDataset(ids), batch_size=BATCH_SIZE,
                     shuffle=(name == 'train'), collate_fn=collate)
    for name, ids in split.items()
}

## 5. Load Pythia (frozen) and define the reconstructor

In [ ]:
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
for p in base.parameters():
    p.requires_grad_(False)

HIDDEN = base.config.hidden_size
print(f'base model loaded; hidden_size={HIDDEN}, layer used={LAYER}')

In [ ]:
class Reconstructor(nn.Module):
    """AR: text z → last-token activation at layer LAYER → learned Linear → ĥ."""
    def __init__(self, base_model, layer: int, hidden: int):
        super().__init__()
        self.base = base_model           # frozen
        self.layer = layer
        self.captured = {}
        target_module = base_model.gpt_neox.layers[layer]
        self._handle = target_module.register_forward_hook(self._hook)
        self.head = nn.Linear(hidden, hidden)
        # Initialise head close to identity — speeds up convergence and gives a
        # sensible starting point if the base layer output is already informative.
        with torch.no_grad():
            self.head.weight.copy_(torch.eye(hidden))
            self.head.bias.zero_()

    def _hook(self, module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        self.captured['h'] = h  # (B, T, D)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            _ = self.base(input_ids=input_ids, attention_mask=attention_mask)
        h_seq = self.captured['h']                       # (B, T, D)
        last_idx = attention_mask.sum(dim=1) - 1         # (B,) — index of last real token
        batch_idx = torch.arange(h_seq.size(0), device=h_seq.device)
        h_last = h_seq[batch_idx, last_idx]              # (B, D)
        return self.head(h_last)                         # (B, D) = ĥ

ar = Reconstructor(base, LAYER, HIDDEN).to(device)
trainable = sum(p.numel() for p in ar.head.parameters())
print(f'reconstructor head trainable params: {trainable:,}')

## 6. FVE & cosine helpers

FVE on **raw** activations (denominator ≈ 999 from notebook 01 — meaningfully large).

In [ ]:
def fve(h_pred: torch.Tensor, h_true: torch.Tensor) -> float:
    mse = ((h_pred - h_true) ** 2).sum(dim=1).mean()
    var = ((h_true - h_true.mean(0)) ** 2).sum(dim=1).mean()
    return (1.0 - mse / var).item()

def mean_cosine(h_pred: torch.Tensor, h_true: torch.Tensor) -> float:
    return F.cosine_similarity(h_pred, h_true, dim=1).mean().item()

## 7. Training

Only the linear head trains (the base model is frozen). Per-epoch we report train loss, val MSE, val FVE, val cosine.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    ar.eval()
    preds, trues = [], []
    for batch in loader:
        enc = tok(batch['z'], padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask).cpu()
        preds.append(h_hat); trues.append(batch['h'])
    p = torch.cat(preds); t = torch.cat(trues)
    mse_full = ((p - t) ** 2).sum(dim=1).mean().item()
    return {'mse': mse_full, 'fve': fve(p, t), 'cos': mean_cosine(p, t)}

optim = torch.optim.AdamW(ar.head.parameters(), lr=LR)
history = []

for epoch in range(1, EPOCHS + 1):
    ar.train()
    running, steps = 0.0, 0
    for batch in loaders['train']:
        enc = tok(batch['z'], padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask)
        h_target = batch['h'].to(device)
        loss = F.mse_loss(h_hat, h_target, reduction='mean') * HIDDEN  # match ||·||²/N convention
        optim.zero_grad(); loss.backward(); optim.step()
        running += loss.item(); steps += 1
    train_mse = running / steps
    val = evaluate(loaders['val'])
    history.append({'epoch': epoch, 'train_mse': train_mse, **{f'val_{k}': v for k, v in val.items()}})
    print(f"epoch {epoch}: train_mse={train_mse:8.3f}  val_mse={val['mse']:8.3f}  "
          f"val_FVE={val['fve']:+.4f}  val_cos={val['cos']:+.4f}")

## 8. Test-set evaluation

The headline numbers.

In [ ]:
test = evaluate(loaders['test'])
print('====  TEST  ====')
print(f"  MSE   = {test['mse']:.4f}")
print(f"  FVE   = {test['fve']:+.4f}")
print(f"  cos   = {test['cos']:+.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(epochs, [h['train_mse'] for h in history], label='train MSE')
axes[0].plot(epochs, [h['val_mse']   for h in history], label='val MSE')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE'); axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(epochs, [h['val_fve'] for h in history], label='val FVE', color='tab:green')
axes[1].plot(epochs, [h['val_cos'] for h in history], label='val cos', color='tab:orange')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('metric'); axes[1].legend(); axes[1].set_title('Validation metrics')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_reconstructor_baseline_training.png', dpi=120)
plt.show()

## 9. Save the trained head

Just the linear head (~590k params). The frozen base is recoverable from Hugging Face.

In [ ]:
out = CKPT_DIR / 'reconstructor_head_baseline.pt'
torch.save({
    'state_dict': ar.head.state_dict(),
    'config': {
        'model_name': MODEL_NAME, 'layer': LAYER, 'hidden': HIDDEN,
        'desc_tokens': DESC_TOKENS, 'epochs': EPOCHS, 'lr': LR, 'seed': SEED,
    },
    'test_metrics': test,
    'history': history,
}, out)
print('saved', out)

## 10. (Optional) commit the figure back to GitHub

In [ ]:
# Uncomment + edit if pushing from Colab.
# !git config --global user.email 'sppooja225@gmail.com'
# !git config --global user.name 'Pooja Sollapura'
# !git add figures/02_reconstructor_baseline_training.png
# !git commit -m 'Baseline reconstructor training curves'
# !git push